# N-Candidate Solver — Concealed Hinges (N=2)

The `NCandidateSolver` solving a **paired-product family** against the real catalog: 53 hinges × 55 mounting plates = 2,915 pairs, 14 constraint rules.

**Compare with:**
- `v2_drawer_slide_demo.ipynb` — N=1 (single product)
- `v2_n_candidate_demo.ipynb` — N=3 (LED lighting)
- `../v1/v1_hinge_constraint_demo.ipynb` — V1 dedicated hinge engine (same data, different solver)

In [ ]:
import sys, time
from pathlib import Path
from collections import Counter

# Find project root
_project_root = Path.cwd()
while _project_root != _project_root.parent:
    if (_project_root / "sample-data").exists() and (_project_root / "engine_v2").exists():
        break
    _project_root = _project_root.parent
sys.path.insert(0, str(_project_root))
DATA_DIR = _project_root / "sample-data"

from engine_v2.core.solver_n import NCandidateSolver
from engine_v2.families.concealed_hinge.config import HINGE_N_CONFIG
from engine_v2.families.concealed_hinge.loader import load_from_json
from engine_v2.families.concealed_hinge.models import (
    ApplicationType, CabinetPosition, CabinetType, HingeRequirements,
)

hinges, plates = load_from_json(DATA_DIR)

solver = NCandidateSolver(
    config=HINGE_N_CONFIG,
    product_lists={"hinge": hinges, "plate": plates},
)

print(f"Loaded {len(hinges)} hinges and {len(plates)} plates")
print(f"Roles: {HINGE_N_CONFIG.role_names}  (N={len(HINGE_N_CONFIG.roles)})")
print(f"Rules: {len(HINGE_N_CONFIG.rules)}")

## 1. Catalog Overview

In [ ]:
brand_counts = Counter(h.brand for h in hinges)
app_counts = Counter(h.application.value for h in hinges)

print("HINGE CATALOG SUMMARY")
print(f"  By brand:       {dict(brand_counts)}")
print(f"  By application: {dict(app_counts)}")
print(f"\nPLATE CATALOG SUMMARY")
print(f"  By brand:       {dict(Counter(p.brand for p in plates))}")
print(f"  By type:        {dict(Counter(p.plate_type.value for p in plates))}")

## 2. Customer Scenarios

Five scenarios exercising different parts of the constraint space.

In [ ]:
scenarios = [
    ("Scenario 1: Standard kitchen — Blum, full overlay, soft-close",
     HingeRequirements(
         cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
         door_weight_kg=5.2, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
         boring_pattern_mm=45, soft_close=True, preferred_brand="Blum")),
    ("Scenario 2: Corner cabinet — needs wide-angle hinge",
     HingeRequirements(
         cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=800,
         door_weight_kg=4.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
         boring_pattern_mm=45, soft_close=True, cabinet_position=CabinetPosition.CORNER)),
    ("Scenario 3: Tall pantry door — 1600mm, 14kg, Grass",
     HingeRequirements(
         cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=22, door_height_mm=1600,
         door_weight_kg=14.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
         boring_pattern_mm=45, soft_close=True, preferred_brand="Grass")),
    ("Scenario 4: Adjacent doors — half overlay, shared partition",
     HingeRequirements(
         cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
         door_weight_kg=4.0, application=ApplicationType.HALF_OVERLAY, desired_overlay_mm=6,
         boring_pattern_mm=45, soft_close=True, preferred_brand="Blum",
         has_adjacent_door=True, adjacent_door_overlay_mm=6, partition_thickness_mm=19)),
    ("Scenario 5: CONSTRAINT VIOLATION — heavy corner door (800mm, 12kg, Blum)",
     HingeRequirements(
         cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=22, door_height_mm=800,
         door_weight_kg=12.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
         boring_pattern_mm=45, soft_close=True, cabinet_position=CabinetPosition.CORNER,
         preferred_brand="Blum")),
]

for title, req in scenarios:
    result = solver.solve_with_explanation(req)
    print("=" * 80)
    print(f"  {title}")
    print(f"  Status: {result['status']}")
    print(f"  {result['message']}")
    if result["status"] == "solved":
        rec = result["recommended"]
        h_sku = rec["candidates"]["hinge"]["sku"]
        p_sku = rec["candidates"]["plate"]["sku"]
        price = f"${rec['total_price_usd']:.2f}" if rec['total_price_usd'] else "N/A"
        print(f"\n  Recommended: {h_sku} + {p_sku}")
        print(f"  Hinges/door: {rec['derived'].get('hinges_per_door', '?')}  |  Price: {price}/door")
        print(f"  + {len(result['alternatives'])} alternative(s)")
    elif result["status"] == "no_solution":
        print(f"\n  Failed rules:")
        for fr in result["failed_rules"]:
            print(f"    [{fr['rule']}] {fr['name']}: {fr['detail']}")
    print()

## 3. Constraint Trace

Full 14-rule trace for Scenario 1's recommended configuration.

In [ ]:
req = scenarios[0][1]
result = solver.solve_with_explanation(req)

if result["status"] == "solved":
    trace = result["recommended"]["constraint_trace"]
    hinge = result["recommended"]["candidates"]["hinge"]
    plate = result["recommended"]["candidates"]["plate"]
    print(f"Configuration: {hinge['sku']} + {plate['sku']}\n")
    print(f"{'Rule':<6} {'Name':<28} {'Result':<6} Detail")
    print("-" * 90)
    for rule in trace:
        icon = "PASS" if rule["passed"] else "FAIL"
        print(f"{rule['rule']:<6} {rule['name']:<28} {icon:<6} {rule['detail']}")

## 4. All Valid Configurations — No Brand Preference

In [ ]:
open_req = HingeRequirements(
    cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
    door_weight_kg=5.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
    boring_pattern_mm=45, soft_close=False,
)

start = time.perf_counter()
all_valid = solver.solve(open_req)
elapsed_ms = (time.perf_counter() - start) * 1000

print(f"Found {len(all_valid)} valid configurations in {elapsed_ms:.1f}ms\n")

brand_dist = Counter(c.candidates["hinge"].brand for c in all_valid)
print(f"By brand: {dict(brand_dist)}\n")

print(f"{'#':>3}  {'Hinge SKU':<22} {'Plate SKU':<22} {'Brand':<8} {'Hinges':>6} {'Price/Door':>11}")
print("-" * 75)
for i, config in enumerate(all_valid[:20], 1):
    h = config.candidates["hinge"]
    p = config.candidates["plate"]
    price = f"${config.total_price_usd:.2f}" if config.total_price_usd else "N/A"
    print(f"{i:>3}  {h.sku:<22} {p.sku:<22} {h.brand:<8} {config.derived['hinges_per_door']:>6} {price:>11}")

if len(all_valid) > 20:
    print(f"\n... and {len(all_valid) - 20} more")

## 5. Pre-filter Impact

In [ ]:
test_reqs = [
    ("Full overlay, frameless, no brand", HingeRequirements(
        cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
        door_weight_kg=5.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
        boring_pattern_mm=45, soft_close=False)),
    ("Full overlay, frameless, Blum", HingeRequirements(
        cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
        door_weight_kg=5.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
        boring_pattern_mm=45, soft_close=False, preferred_brand="Blum")),
    ("Corner, frameless, no brand", HingeRequirements(
        cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=800,
        door_weight_kg=4.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
        boring_pattern_mm=45, soft_close=False, cabinet_position=CabinetPosition.CORNER)),
]

full_product = len(hinges) * len(plates)
print(f"Full Cartesian product: {len(hinges)} x {len(plates)} = {full_product:,} pairs\n")
print(f"{'Scenario':<40} {'Filtered hinges':>16} {'Pairs evaluated':>16} {'Reduction':>10}")
print("-" * 85)

for label, req in test_reqs:
    filtered = solver._apply_pre_filters(req)
    n_hinges = len(filtered["hinge"])
    n_pairs = n_hinges * len(filtered["plate"])
    reduction = (1 - n_pairs / full_product) * 100
    print(f"{label:<40} {n_hinges:>16} {n_pairs:>16,} {reduction:>9.0f}%")

## 6. Performance

In [ ]:
bench_scenarios = [
    ("Standard (Blum, full overlay)", scenarios[0][1]),
    ("Corner cabinet (all brands)", scenarios[1][1]),
    ("Tall pantry (Grass)", scenarios[2][1]),
    ("Adjacent doors (Blum, half)", scenarios[3][1]),
    ("No brand preference", open_req),
]

print(f"{'Scenario':<35} {'Time (ms)':>10} {'Valid':>6} {'Pairs evaluated':>16}")
print("-" * 70)
for label, req in bench_scenarios:
    filtered = solver._apply_pre_filters(req)
    n_pairs = len(filtered["hinge"]) * len(filtered["plate"])
    start = time.perf_counter()
    results = solver.solve(req)
    elapsed = (time.perf_counter() - start) * 1000
    print(f"{label:<35} {elapsed:>9.1f}ms {len(results):>6} {n_pairs:>16,}")